# 초구 예측 — model_09 (2단계: 구종 LightGBM 보정을 월 단위로 다시 학습)

- 0장 = 모델 전 확인
  - 0-1 투수 유형 변수(배합 다양성·주 구종 비율): 효과 없음, 적용 안 함 (README 4-18)
  - 0-2 재학습 주기: 시즌 후반에 보정 이득이 줄지 않음 = 상황 효과는 시즌 안에서 낡지 않음 (README 4-19)
  - 0-3 월 단위 재학습 직접 시험: 2019 −0.0008, 2020 −0.0010 (둘 다 95% 구간 0 위)
- 모델: model_07에서 **LightGBM 보정의 재학습 주기만** 시즌 단위 -> 월 단위로 바꿈 (사용자 결정 2026-11-10)
  - 그 달 첫 경기 전날까지의 모든 초구로 다시 학습 (지난 시즌들 + 그 시즌 지난달까지)
  - B4·변수·LightGBM 파라미터·트리 수·위치는 model_07 그대로
- 비교 대상: model_07

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_09"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

- 방법 (model_08 0장과 같음): model_07 구조(B4 출발 LightGBM, 변수 `FEAT`, 파라미터 model_07 값) 위에 새 변수를 더해 2017–18 학습 -> 2019
  - model_07 구조만 대비 2019 log loss 이득 × 1000 (양수 = 좋아짐), 경기 단위 부트스트랩, 원래 + 기록 100개 미만 공 뺌 (2017 출발점 문제, README 4-11)

In [2]:
# 공통: model_07 구조 (B4 + FEAT), 비교용 LightGBM
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)  # model_06·07 B4
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]  # model_07 변수

def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06·07 B4 (n, 9)
    first = year == fp["GYEAR"].min()
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

NC = len(CLASSES)
P_B4 = np.full((len(fp), NC), np.nan)
for yr in range(2017, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
assert abs(fc.prob_metrics(P_B4[val], y_type[val])["logloss"] - 1.09367) < 1e-5  # model_06·07 B4 재현
XF = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)  # 0장 변수 표 (FEAT + 새 변수)
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)
GID = fp["gameID"].to_numpy()
yr_all = fp["GYEAR"].to_numpy()
i19 = np.flatnonzero(val)
_, N_raw = fc.lookup(fc.prior_table(fp, ["pitcher_code"], CLASSES[:1]), fp, ["pitcher_code"], CLASSES[:1])
n_prior = N_raw[:, 0]  # 전날까지 그 투수 초구 수
TRAINS = {"원래 (2017–18)": np.flatnonzero((yr_all >= 2017) & (yr_all <= 2018)),
          "기록 100개 미만 공 뺌": np.flatnonzero((yr_all >= 2017) & (yr_all <= 2018) & (n_prior >= 100))}
SCREEN_P = dict(objective="multiclass", num_class=NC, learning_rate=0.03, num_leaves=7, min_data_in_leaf=20, lambda_l2=1000.0,
                feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, verbose=-1, seed=0, deterministic=True,
                force_row_wise=True, num_threads=4)  # model_07 값

def screen_ll(cols, tidx):  # 학습 행 tidx(경기 20%는 조기 종료용)로 학습 -> 2019 공별 log loss
    g = np.unique(GID[tidx])
    e = np.isin(GID[tidx], np.random.default_rng(1).choice(g, len(g) // 5, replace=False))
    t_, v_ = tidx[~e], tidx[e]
    dtr = lgb.Dataset(XF[cols].iloc[t_], y_all[t_], init_score=INIT[t_], free_raw_data=False)
    dva = lgb.Dataset(XF[cols].iloc[v_], y_all[v_], init_score=INIT[v_], reference=dtr)
    bst = lgb.train(SCREEN_P, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(XF[cols].iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + INIT[i19]
    return -log_softmax(raw, axis=1)[np.arange(len(i19)), y_all[i19]]

def game_ci(d, gid=None, B=1000):  # 경기 단위 부트스트랩 95% 구간
    s = pd.DataFrame({"g": GID[i19] if gid is None else gid, "d": d}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

def screen(groups):  # 묶음별: model_07 구조(FEAT)만 대비 이득 표, 두 방법
    all_cols = sum(groups.values(), [])
    rows = []
    for sname, tidx in TRAINS.items():
        base = screen_ll(FEAT, tidx)
        for nm, cols in {"전부": all_cols, **groups}.items():
            d = base - screen_ll(FEAT + cols, tidx)
            lo, hi = game_ci(d)
            rows.append({"방법": sname, "더한 변수": nm, "model_07 구조 2019 log loss": base.mean(),
                         "이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000})
    return pd.DataFrame(rows).set_index(["방법", "더한 변수"]).round(4)
print(f"2019: B4 {fc.prob_metrics(P_B4[val], y_type[val])['logloss']:.4f}, model_07 {fc.prob_metrics(fc.align(pd.read_parquet('results/model_07/per_pitch.parquet'), fp, fc.VAL_YEAR)[fc.P_TYPE].to_numpy(), y_type[val])['logloss']:.4f}")

2019: B4 1.0937, model_07 1.0621


### 0-1. 투수 유형: 배합 다양성(엔트로피)·주 구종 비율

- 기대: 트리가 "구종 한두 개인 투수"와 "고르게 섞는 투수"에서 상황 효과를 다르게 적용 (지금은 투수 이름도 유형도 모름)
- 두 가지 출처 (모두 전날까지)
  - 초구 배합 (B4 확률): 엔트로피 −Σ p·log p, 주 구종 비율 = 가장 큰 확률
  - 모든 카운트 배합 (`fc.pitcher_history_features` mix_all, 반감 200구, C에서 미래 정보 검사 통과): 엔트로피, 주 구종 비율 (기록 없으면 빈 값)
- 범위가 해마다 같은 값 (0–log 9, 0–1) -> `bat_n` 문제 없음, 2017 초반 B4는 리그 비율이라 엔트로피가 높음 -> 두 방법

In [3]:
ent = lambda P: -(np.clip(P, 1e-12, None) * np.log(np.clip(P, 1e-12, None))).sum(1)  # 배합 엔트로피 (n,)
H = fc.pitcher_history_features(m, fp)  # 모든 카운트 배합 (전날까지)
MIX = H[["mix_all_" + c for c in CLASSES]].to_numpy()
XF["ty_ent_b4"], XF["ty_top_b4"] = ent(P_B4), P_B4.max(1)
has = ~np.isnan(MIX).any(1)
XF["ty_ent_all"] = np.where(has, ent(np.nan_to_num(MIX)), np.nan)
XF["ty_top_all"] = np.where(has, np.nanmax(np.where(has[:, None], MIX, 0), 1), np.nan)
TY = ["ty_ent_b4", "ty_top_b4", "ty_ent_all", "ty_top_all"]
display(pd.DataFrame({y: XF.loc[yr_all == y, TY].mean() for y in (2017, 2018, 2019)}).round(3).rename_axis("해별 평균"))
display(XF.loc[val, TY].describe().T[["mean", "std", "min", "max"]].round(3))
TY_GROUPS = {"초구 배합 (B4) 기준": ["ty_ent_b4", "ty_top_b4"], "모든 카운트 배합 기준": ["ty_ent_all", "ty_top_all"]}
display(screen(TY_GROUPS))

,2017,2018,2019
해별 평균,,,
ty_ent_b4,1.072,1.073,1.076
ty_top_b4,0.559,0.546,0.533
ty_ent_all,1.095,1.124,1.151
ty_top_all,0.549,0.525,0.509


,mean,std,min,max
ty_ent_b4,1.076,0.214,0.275,1.642
ty_top_b4,0.533,0.112,0.280,0.940
ty_ent_all,1.151,0.181,0.579,1.672
ty_top_all,0.509,0.083,0.255,0.806


model_07 구조 2019 log loss  이득 ×1000  95% 하한  95% 상한
방법             더한 변수                                                             
원래 (2017–18)   전부                                1.0635   -2.8154 -5.2201 -0.5888
               초구 배합 (B4) 기준                     1.0635   -1.3435 -3.0594  0.4523
               모든 카운트 배합 기준                      1.0635   -2.4336 -3.8246 -1.0427
기록 100개 미만 공 뺌 전부                                1.0654    0.3188 -1.4964  2.0354
               초구 배합 (B4) 기준                     1.0654    0.6796 -0.6128  1.9926
               모든 카운트 배합 기준                      1.0654   -0.4884 -1.8182  0.7502

#### 🔍 0-1 결과
#### 투수 유형(배합 다양성·주 구종 비율)은 model_07 위에 남은 효과 없음 -> 적용 안 함
- 원래: 전부 −2.8 (나빠짐), 초구 배합(B4) 기준 −1.3, 모든 카운트 배합 기준 −2.4 (나빠짐)
- 기록 100개 미만 공 뺌: +0.3 · +0.7 · −0.5 (모두 95% 구간에 0 포함) -> 2017 문제를 걸러도 이득 없음
- 해마다 범위가 조금씩 움직임: 리그 배합이 다양해짐 (모든 카운트 엔트로피 2017 1.095 -> 2018 1.124 -> 2019 1.151, 주 구종 비율 0.549 -> 0.509)
- 엔트로피·주 구종 비율은 B4 확률에서 바로 나오는 값 -> 출발점(log B4)에 이미 있는 정보, 유형별로 상황 효과를 다르게 배울 신호는 없음 (model_08 0-1 투수×상황 표본 부족과 이어짐)


### 0-2. 재학습 주기: 시즌 후반 성능 하락 확인

- 지금 구조: B4(투수 배합)는 매일 갱신, LightGBM 보정(상황 효과)은 시즌에 한 번 학습 (2019 = 2017–18 학습, 2020 = 2017–19 학습) -> 시즌 안에서는 고정
- 질문: 보정 이득(B4 log loss − model_07 log loss)이 시즌 후반으로 갈수록 줄어드는가 -> 줄어들면 시즌 초에 배운 상황 효과가 낡아 간다는 신호 = 월 단위 재학습 시험할 이유
- 방법: model_07 결과 파일(`results/model_07/per_pitch.parquet`)과 B4를 월별로 비교 (새 학습 없음)
  - 3월은 4월에, 10월은 9월에 합침 (경기 수가 적음), 경기 단위 부트스트랩
  - 판단은 2019, 2020은 확인만
- 직구 비율(실제 / model_07 평균 / B4 평균)도 같이 봄: 한쪽으로 쏠리는 달이 있는지

In [3]:
pp11 = pd.read_parquet("results/model_07/per_pitch.parquet")
F = CLASSES.index("FAST")
mrows = []
for yr, mask in ((fc.VAL_YEAR, val), (fc.TEST_YEAR, test)):
    q = fp[mask]
    PM, PB, y = fc.align(pp11, fp, yr)[fc.P_TYPE].to_numpy(), P_B4[mask], y_type[mask]
    ii = np.arange(len(y))
    mrows.append(pd.DataFrame({"연도": yr, "월": q["gameID"].str[4:6].astype(int).clip(4, 9).to_numpy(),  # 3월 -> 4월, 10월 -> 9월
                               "g": q["gameID"].to_numpy(),
                               "b4": -np.log(np.clip(PB[ii, y], 1e-12, None)), "m07": -np.log(np.clip(PM[ii, y], 1e-12, None)),
                               "fast": y == F, "fast_m11": PM[:, F], "fast_b4": PB[:, F]}))
MD = pd.concat(mrows)
MD["gain"] = MD["b4"] - MD["m07"]

tab = []
for (yr, mon), d in MD.groupby(["연도", "월"]):
    lo, hi = game_ci(d["gain"].to_numpy(), d["g"].to_numpy())
    tab.append({"연도": yr, "월": mon, "공": len(d), "B4 log loss": d["b4"].mean(), "model_07 log loss": d["m07"].mean(),
                "보정 이득 ×1000": d["gain"].mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                "직구 실제": d["fast"].mean(), "직구 model_07": d["fast_m11"].mean(), "직구 B4": d["fast_b4"].mean()})
display(pd.DataFrame(tab).set_index(["연도", "월"]).round(4))

def late_minus_early(d, B=1000):  # 후반(7–9월) 이득 − 전반(4–6월) 이득, 경기 단위 부트스트랩
    s = d.groupby("g").agg(s=("gain", "sum"), n=("gain", "size"), late=("월", "first"))
    s["late"] = s["late"] >= 7
    rng, res = np.random.default_rng(0), []
    for _ in range(B):
        b = s.iloc[rng.integers(0, len(s), len(s))]
        L, E = b[b["late"]], b[~b["late"]]
        res.append(L["s"].sum() / L["n"].sum() - E["s"].sum() / E["n"].sum())
    return np.percentile(np.array(res) * 1000, [2.5, 97.5])

half = []
for yr, d in MD.groupby("연도"):
    late = d["월"] >= 7
    lo, hi = late_minus_early(d)
    half.append({"연도": yr, "전반(4–6월) 이득 ×1000": d["gain"][~late].mean() * 1000, "후반(7–9월) 이득 ×1000": d["gain"][late].mean() * 1000,
                 "후반 − 전반": (d["gain"][late].mean() - d["gain"][~late].mean()) * 1000, "95% 하한": lo, "95% 상한": hi})
display(pd.DataFrame(half).set_index("연도").round(2))

공  B4 log loss  model_07 log loss  보정 이득 ×1000   95% 하한   95% 상한   직구 실제  직구 model_07   직구 B4
연도   월                                                                                                  
2019 4  2568       1.1004             1.0701      30.3055  22.9088  38.8688  0.5058       0.5087  0.5110
     5  2403       1.1223             1.0925      29.7795  20.5396  38.7438  0.4873       0.4902  0.4945
     6  2493       1.1580             1.1321      25.9297  16.5782  35.8659  0.4789       0.4971  0.5018
     7  1791       1.0991             1.0603      38.8031  28.4039  50.0271  0.4830       0.4781  0.4791
     8  2362       1.0333             1.0019      31.4185  20.2725  42.4476  0.4903       0.4818  0.4814
     9  1841       1.0320             0.9956      36.4533  26.9042  45.8122  0.5106       0.4965  0.4941
2020 5  2264       1.1800             1.1405      39.5579  28.5976  50.4449  0.4541       0.4587  0.4582
     6  2379       1.1310             1.0961      34.9038  24.5472  45.4233  0.4279       0.4552  0.4524
     7  2233       1.0621             1.0258      36.3389  27.5709  43.7616  0.4962       0.4639  0.4659
     8  1985       1.0407             1.0124      28.2697  18.3843  38.0414  0.5058       0.4819  0.4793
     9  3754       1.0577             1.0241      33.5236  26.7015  40.2362  0.5032       0.4992  0.4990

,전반(4–6월) 이득 ×1000,후반(7–9월) 이득 ×1000,후반 − 전반,95% 하한,95% 상한
연도,,,,,
2019,28.67,35.17,6.50,-1.64,14.31
2020,37.17,33.00,-4.17,-13.24,5.09


#### 🔍 0-2 결과
#### 시즌 후반에 보정 이득이 줄어들지 않음 = 상황 효과는 시즌 안에서 낡지 않음 (그 시즌 데이터를 더 넣는 몫은 0-3에서 따로 확인)
- 2019 (판단): 보정 이득 ×1000이 4–6월 30.3 · 29.8 · 25.9 -> 7–9월 38.8 · 31.4 · 36.5, 후반 − 전반 = +6.5 (95% −1.6 ~ +14.3) -> 오히려 후반이 조금 더 좋음 (구간에 0 포함)
- 2020 (확인): 5–6월 39.6 · 34.9 -> 7–9월 36.3 · 28.3 · 33.5, 후반 − 전반 = −4.2 (95% −13.2 ~ +5.1) -> 구간에 0 포함, 하락 신호라고 보기 어려움
- 달마다 이득이 20–40 사이로 움직이지만 방향이 없음 = 달별 잡음 수준 (한 달 약 2,000구)
- 직구 비율: 2019 6월(실제 0.479, 예측 0.497), 2020 6월(0.428 / 0.455)·7월(0.496 / 0.464)처럼 한 달 단위로 몇 %p 어긋나는 달이 있지만 model_07과 B4가 같이 어긋남 -> 상황 보정이 아니라 투수 배합(B4) 쪽 흔들림, B4는 이미 매일 갱신
- 결론: 상황 효과는 시즌 안에서 낡지 않음 -> 낡은 효과를 고치려고 월 단위로 다시 학습할 이유는 없음. 다만 이득 차이 구간이 ±10 정도로 넓어 작은 하락(×1000 기준 몇)은 못 잡음

### 0-3. 월 단위 재학습 직접 시험

- 0-2에서 후반 하락은 없었지만, 월 단위 재학습에는 "그 시즌 지난달까지 데이터를 학습에 더 넣는" 이득이 따로 있을 수 있음 -> 직접 확인 (사용자 요청)
- 시즌 단위 (= model_07): 2019 전체를 2017–18로 학습한 모델 하나로 예측
- 월 단위: 2019 각 달(3월은 4월에, 10월은 9월에 합침)을 **그 달 첫날 전까지**의 모든 초구(2017–18 + 2019 지난달까지)로 다시 학습해 예측
  - 변수 `FEAT`, 파라미터·트리 수는 model_07 값 그대로 (잎 7, min_data 20, λ 1000, 트리 730) -> 재학습 주기만 바꿈
  - 조기 종료 없음 (미래 달을 쓰지 않도록)
- 판단은 2019, 2020은 같은 방법(2017–19 + 2020 지난달까지)으로 확인만

In [3]:
LGB_P11 = dict(SCREEN_P, num_leaves=7, min_data_in_leaf=20, lambda_l2=1000.0)  # model_07 값
ROUNDS11 = 730
X11 = XF[FEAT]
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 3월 -> 4월, 10월 -> 9월

def fit_pred(tr_idx, te_idx):  # tr_idx로 학습 -> te_idx 구종 확률
    bst = lgb.train(LGB_P11, lgb.Dataset(X11.iloc[tr_idx], y_all[tr_idx], init_score=INIT[tr_idx]), ROUNDS11)
    return np.exp(log_softmax(bst.predict(X11.iloc[te_idx], raw_score=True) + INIT[te_idx], axis=1))

res = {}
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    P_season = fit_pred(np.flatnonzero(yr_all < yr), iy)  # 시즌 단위
    P_month = np.full_like(P_season, np.nan)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all < yr) | ((yr_all == yr) & (DATE < start)))
        assert DATE[tr].max() < start
        P_month[np.searchsorted(iy, te)] = fit_pred(tr, te)
        print(f"{yr} {mon}월: 학습 초구 {len(tr):,} (그 시즌 {((yr_all[tr] == yr)).sum():,})")
    res[yr] = (iy, P_season, P_month)
P11 = {yr: fc.align(pd.read_parquet("results/model_07/per_pitch.parquet"), fp, yr)[fc.P_TYPE].to_numpy() for yr in res}
assert np.allclose(res[fc.VAL_YEAR][1], P11[fc.VAL_YEAR], atol=1e-6) and np.allclose(res[fc.TEST_YEAR][1], P11[fc.TEST_YEAR], atol=1e-6)  # 시즌 단위 = model_07 재현

rows, mrows = [], []
for yr, (iy, Ps, Pm) in res.items():
    y = y_all[iy]
    lls = -np.log(Ps[np.arange(len(y)), y]); llm = -np.log(Pm[np.arange(len(y)), y])
    d = lls - llm  # 양수 = 월 단위가 좋음
    lo, hi = game_ci(d, GID[iy])
    rows.append({"연도": yr, "시즌 단위 log loss": lls.mean(), "월 단위 log loss": llm.mean(),
                 "월 단위 이득 ×1000": d.mean() * 1000, "95% 하한": lo * 1000, "95% 상한": hi * 1000,
                 "top1 시즌": (Ps.argmax(1) == y).mean(), "top1 월": (Pm.argmax(1) == y).mean()})
    for mon in np.unique(MON[iy]):
        k = MON[iy] == mon
        mrows.append({"연도": yr, "월": mon, "공": k.sum(), "월 단위 이득 ×1000": d[k].mean() * 1000})
display(pd.DataFrame(rows).set_index("연도").round(4))
display(pd.DataFrame(mrows).set_index(["연도", "월"]).round(2).unstack(0))

2019 4월: 학습 초구 24,636 (그 시즌 0)
2019 5월: 학습 초구 27,204 (그 시즌 2,568)
2019 6월: 학습 초구 29,607 (그 시즌 4,971)
2019 7월: 학습 초구 32,100 (그 시즌 7,464)
2019 8월: 학습 초구 33,891 (그 시즌 9,255)
2019 9월: 학습 초구 36,253 (그 시즌 11,617)
2020 5월: 학습 초구 38,094 (그 시즌 0)
2020 6월: 학습 초구 40,358 (그 시즌 2,264)
2020 7월: 학습 초구 42,737 (그 시즌 4,643)
2020 8월: 학습 초구 44,970 (그 시즌 6,876)
2020 9월: 학습 초구 46,955 (그 시즌 8,861)


,시즌 단위 log loss,월 단위 log loss,월 단위 이득 ×1000,95% 하한,95% 상한,top1 시즌,top1 월
연도,,,,,,,
2019,1.0621,1.0613,0.8088,0.0595,1.5401,0.538,0.5385
2020,1.0570,1.0561,0.9520,0.3758,1.5310,0.534,0.5332


공         월 단위 이득 ×1000      
연도    2019    2020          2019  2020
월                                     
4   2568.0     NaN          0.00   NaN
5   2403.0  2264.0          0.41  0.00
6   2493.0  2379.0          0.35  0.78
7   1791.0  2233.0         -0.42  1.33
8   2362.0  1985.0          3.64  1.31
9   1841.0  3754.0          0.64  1.22

#### 🔍 0-3 결과
#### 월 단위 재학습이 작지만 두 해 모두 일관되게 좋음: 2019 −0.0008, 2020 −0.0010 (95% 구간 모두 0 위)
- 시즌 단위는 model_07과 확률이 같음 (재현 확인)
- 2019 (판단): 1.0621 -> 1.0613, 이득 ×1000 +0.8 (95% +0.1 ~ +1.5)
- 2020 (확인): 1.0570 -> 1.0561, +1.0 (95% +0.4 ~ +1.5) -> 나빠지지 않음
- 달별: 첫 달은 학습 데이터가 같아 0, 그 뒤 대부분 0.4–3.6 (2019 7월만 −0.4) -> 그 시즌 데이터가 쌓일수록 조금씩 나아짐
- 0-2와 이어서: 상황 효과가 낡아서가 아니라, 그 시즌 리그 경향(공 수 약 2,000–12,000개)을 학습에 더 넣는 몫
- 크기: model_07 앞 타석 초구 결과(−0.0003)보다 크고 시즌 초 배수(−0.0036)보다 작음
- top1은 거의 같음 (2019 53.80% -> 53.85%, 2020 53.40% -> 53.32%)
- 트리 수 730은 2017–18 학습 기준 -> 데이터가 늘면 더 많은 트리가 맞을 수 있지만 고정 (미래 달 없이 고를 방법이 없음)

## 모델

In [3]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 모두 model_07 값 고정 (구종 B4, 변수 `FEAT`, LightGBM 잎 7·잎당 최소 20·λ 1000·트리 730, 위치는 model_03)
- 바꾸는 것: LightGBM 재학습 주기 `RETRAIN = "month"` 하나
  - 트리 수는 다시 고르지 않음: 달마다 고르려면 미래 달이 필요하고, 2019 전체로 다시 고르면 2019 판단에 쓰는 셈

In [4]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 (model_03, 고정)
LOC_REF, LOC_REF_HL = "decay", 173.0
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 17.3
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM (월 단위 재학습)

### 2-1. B4 (model_06·07과 같음)

In [5]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [6]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-3. 월 단위 재학습과 구종 확률

- 해마다 달별로: 학습 = 그 달 첫 경기일보다 앞선 모든 초구 (`DATE < 시작일`) -> 그 달 예측
  - 첫 달(2019 3–4월, 2020 5월)은 그 시즌 데이터가 없어 시즌 단위(model_07)와 같은 모델
  - 3월은 4월에, 10월은 9월에 합침 (경기 수가 적음, 0-3과 같음)
- 확인: 첫 달 확률 = model_07 (재현), 학습 데이터 끝 < 예측 달 시작 (미래 정보 없음)

In [7]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_09    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_09    1.0561  0.5332  0.8218

### 2-4. 진단

- 달별 model_07 대비 이득 (0-3과 같아야 함)
- 구종 1등 보정 구간표: model_07 vs model_09 (월 단위 학습이 확률을 한쪽으로 치우치게 하지 않는지)

In [8]:
nll = lambda P, y: -np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None))
rows = []
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    d = nll(P11[yr], y_type[msk]) - nll(P_TYPE_ALL[msk], y_type[msk])  # 양수 = model_09이 좋음
    for mon in np.unique(MON[msk]):
        k = MON[msk] == mon
        rows.append({"연도": yr, "월": mon, "공": int(k.sum()), "model_07 대비 이득 ×1000": d[k].mean() * 1000})
display(pd.DataFrame(rows).set_index(["연도", "월"])["model_07 대비 이득 ×1000"].unstack(0).round(2))
# 구종 1등 보정 구간표 (2019·2020): model_07 vs model_09
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        rep, tabs = fc.calib_report(P, y_type[msk], CLASSES)
        t = tabs["1등 예측"]
        print(f"{yr} {nm}: 1등 ECE {rep.loc['1등 예측', 'ECE']:.4f}")
        display(t[t["공수"] >= 30].round(3).T)

연도,2019,2020
월,,
4,0.00,NaN
5,0.41,0.00
6,0.35,0.78
7,-0.42,1.33
8,3.64,1.31
9,0.64,1.22


2019 model_07: 1등 ECE 0.0141


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,94.000,1826.000,3854.000,3499.000,2253.000,1274.000,540.000,118.000
예측평균,0.285,0.361,0.453,0.546,0.645,0.744,0.846,0.925
실제비율,0.330,0.377,0.445,0.529,0.630,0.737,0.889,0.966
차이,0.045,0.015,-0.007,-0.017,-0.015,-0.006,0.043,0.042


2019 model_09: 1등 ECE 0.0128


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,92.000,1853.000,3886.000,3442.000,2267.000,1219.000,560.000,139.000
예측평균,0.284,0.362,0.453,0.546,0.646,0.744,0.846,0.925
실제비율,0.304,0.378,0.452,0.525,0.628,0.742,0.871,0.986
차이,0.020,0.017,-0.000,-0.021,-0.018,-0.002,0.026,0.061


2020 model_07: 1등 ECE 0.0142


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,232.000,1813.000,3484.000,3096.000,1956.000,1228.000,680.000,125.000
예측평균,0.271,0.363,0.452,0.547,0.645,0.745,0.842,0.923
실제비율,0.272,0.359,0.442,0.523,0.632,0.741,0.876,0.976
차이,0.001,-0.004,-0.010,-0.025,-0.013,-0.004,0.035,0.053


2020 model_09: 1등 ECE 0.0155


,20%~30%,30%~40%,40%~50%,50%~60%,60%~70%,70%~80%,80%~90%,90%~100%
공수,231.000,1815.000,3452.000,3141.000,1925.000,1225.000,688.000,138.000
예측평균,0.271,0.363,0.452,0.546,0.644,0.744,0.844,0.924
실제비율,0.277,0.364,0.434,0.525,0.626,0.746,0.874,0.978
차이,0.006,0.000,-0.018,-0.021,-0.019,0.002,0.030,0.054


#### 🔍 진단
#### 달별 이득은 0-3과 같음, 확률이 한쪽으로 치우치지 않음
- 달별 model_07 대비 이득 ×1000: 2019 0 · +0.4 · +0.4 · −0.4 · +3.6 · +0.6, 2020 0 · +0.8 · +1.3 · +1.3 · +1.2 (첫 달은 같은 모델이라 0)
- 그 시즌 학습 초구: 2019 9월 11,617개, 2020 9월 8,861개까지 늘어남 (전체 학습의 약 20–30%)
- 구종 1등 ECE: 2019 0.0141 -> 0.0128, 2020 0.0142 -> 0.0155 -> 구간표 모양 거의 같음 (높은 쪽 80% 이상에서 실제가 예측보다 높은 것은 model_07과 같음)


## 3. ② 위치 (model_03과 같음)

In [10]:
# 24칸 적분 함수는 fp_common으로 옮김: fc.zone_from_gauss_cov(g, q, m=32) (계산 설명은 위, 코드 주석은 fp_common.py)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

In [11]:
L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 리그 구종×타자 손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 리그 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 리그 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None):  # 위치 모멘트 + 24칸 비율 (n, 29): L2
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    S1, N1 = fc.lookup(fc.prior_table(fp, K1, L_COLS, half_life), q, K1, L_COLS)
    P = fc.shrink(S1, N1, P, k_type)  # 투수×구종 쪽으로 k_type만큼
    S2, N2 = fc.lookup(fc.prior_table(fp, K2, L_COLS, half_life), q, K2, L_COLS)
    return fc.shrink(S2, N2, P, k_hand)  # 투수×구종×손 쪽으로 k_hand만큼

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, cov=True):  # 구종이 t라고 가정했을 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)  # 구종만 t로 바꾼 사본
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl), qt)  # 모멘트 -> 정규분포 (높이는 그 타자 존으로 cm 환산)
    return zone_probs(g, qt, cov), g  # 정규분포를 24칸 위에서 적분 (cov=True 공분산 반영)

## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_06과 같음, 구종 확률만 2장 결과(`P_TYPE_ALL`)

In [12]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL)
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [13]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종과 무관한 열(실제 구종 기준 위치)은 model_07과 같아야 함
chk = pp.merge(m07, on=fc.KEYS, suffixes=("", "_03"))
same = ["ll_zone_true", "nll_coord_true", "dist_true"]
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_03"], equal_nan=True)]
assert len(chk) == len(pp) and not bad, f"model_07과 다른 열: {bad}"
print("실제 구종 기준 위치 = model_07 (재현 확인)")

(26073, 48)
실제 구종 기준 위치 = model_07 (재현 확인)


## 5. 평가

- 채점은 `fc.evaluate`, model_07과 나란히 / 경기 단위 부트스트랩 (model_09 − model_07, 2020)
- 2020은 확인용 -> 2020을 보고 파라미터 다시 고르지 않음

In [14]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m07_sum = eval_years(m07)
display(pd.concat({MODEL: summary, "model_07": m07_sum, "차이 (model_07 대비)": summary - m07_sum}, axis=1).round(4))

model_09          model_07          차이 (model_07 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0621   1.0570          -0.0008 -0.0010
구종 top1               0.5385   0.5332   0.5380   0.5340           0.0005 -0.0009
구종 top2               0.8177   0.8218   0.8174   0.8209           0.0004  0.0009
구종 top3               0.9374   0.9412   0.9371   0.9413           0.0003 -0.0002
구종 macro F1           0.3226   0.3545   0.3205   0.3510           0.0022  0.0035
24칸 logloss (실전)      2.9493   2.9454   2.9495   2.9456          -0.0002 -0.0001
24칸 top1              0.1300   0.1337   0.1298   0.1340           0.0002 -0.0004
24칸 top3              0.2888   0.2908   0.2893   0.2912          -0.0005 -0.0003
H/S/C logloss         1.0748   1.0767   1.0748   1.0767          -0.0000 -0.0000
H/S/C 정확도             0.3951   0.3896   0.3954   0.3891          -0.0003  0.0005
24칸 logloss (실제 구종)   2.8933   2.8967   2.8933   2.8967           0.0000  0.0000
구종×칸 logloss          3.9546   3.9521   3.9554   3.9530          -0.0008 -0.0010
좌표 평균거리 cm           32.6837  31.8305  32.6877  31.8346          -0.0040 -0.0041
좌표 NLL                9.3171   9.2660   9.3174   9.2662          -0.0002 -0.0002

In [15]:
display(fc.bootstrap_diff(fc.pitch_scores(m07, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
# 실전 출력: 누적 80% 보여 주기 (2020)
for nm, P_ in [("model_07", m07), (MODEL, pp)]:
    d = fc.align(P_, fp, fc.TEST_YEAR)
    Pt_, yt_ = d[fc.P_TYPE].to_numpy(), d["pitch_type"].map(fc.TYPE_IDX).to_numpy()
    s = fc.show_types(Pt_)
    print(f"{nm}: 보여 준 구종 수 평균 {s.sum(1).mean():.2f}, 포함률 {s[np.arange(len(yt_)), yt_].mean():.3f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,-0.0010,-0.0015,-0.0004,좋아짐
ll_zone,-0.0001,-0.0003,-0.0000,좋아짐
ll_joint,-0.0010,-0.0015,-0.0004,좋아짐
nll_coord,-0.0002,-0.0003,-0.0000,좋아짐
dist,-0.0041,-0.0069,-0.0015,좋아짐


model_07: 보여 준 구종 수 평균 2.40, 포함률 0.904
model_09: 보여 준 구종 수 평균 2.39, 포함률 0.905


### 🥇1등🥇
#### 2020 구종 log loss 1.0570 -> 1.0561 (−0.0010), 2019도 −0.0008, 부트스트랩 6개 점수 모두 좋아짐 -> 채택
- 재학습 주기만 바꿈 (B4·변수·파라미터·트리 수·위치는 model_07 그대로)
- 부트스트랩 (2020, model_09 − model_07): 구종 −0.0010, 24칸 −0.0001, 동시 −0.0010, 좌표 NLL −0.0002, 거리 −0.004cm -> 모두 95% 구간 0 아래
- top1은 2019 +0.0005, 2020 −0.0009 (거의 같음), macro F1 +0.002 · +0.0035, 누적 80% 보여 주기 그대로 (평균 2.39개, 포함률 90.5%)
- 이득 출처: 상황 효과가 낡아서가 아니라 (0-2) 그 시즌 지난달까지 데이터를 학습에 더 넣는 몫
- 운영: 매달 첫 경기 전에 그때까지 데이터로 다시 학습 (시즌당 약 6번)
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-10)


## 6. 저장

- `results/model_09/` -> `compare.ipynb` 실행하면 자동 포함

In [16]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_09 ['per_pitch.parquet', 'summary.csv']


## 7. 2단계 끝 구종 파라미터 확인 (사용자 요청 2026-11-10)

- 규칙 (README 3장 "파라미터 다시 고르기"): 단계가 끝나면 바뀐 단계 파라미터를 전부 다시 고름 -> 2단계 마감 때 빠뜨려 여기서 함 (1단계 끝 확인도 마지막 모델 model_03 안에서 했음)
  - 바뀐 단계 = 구종만 (위치는 실제 구종 기준이라 2단계 동안 그대로)
- 지금 값: B4는 model_06에서 **B4 단독 log loss**로 고름, LightGBM은 model_07에서 고름 -> B4는 LightGBM 출발점이라 연결된 파라미터 -> **최종 구종 log loss(2019)** 기준으로 함께 확인
- 방법 (계산을 줄이려고 두 단계)
  - 7-1 B4: 반감기 25·35.4·50 × alpha 5·7.5·11 × hand_k 3.5·5.48·8 (시즌 초 0.25) + 시즌 초 0.15·0.4 (나머지 지금 값), LightGBM은 지금 값 -> 1등이 범위 끝이면 그 방향으로 넓힘
  - 7-2 LightGBM: 7-1 1등 B4 위에서 잎 7·15 × 잎당 최소 20·50·200 × lambda_l2 300·1000·3000 (model_07 2-3과 같은 격자)
  - 채점은 model_07 방식 = 2017–18 학습, 2019로 조기 종료한 트리 수의 2019 log loss (시즌 단위), 지금 값(대조군)도 같은 방식으로 채점
  - 7-3 고른 값과 지금 값을 **월 단위 재학습**(트리 수 고정)으로 2019 비교
- 판단: 지금 값과 차이가 0.0002(model_07 평평한 구간 기준) 안이면 "확인 결과 그대로" -> model_09 그대로 / 확실히 나으면 model_09은 덮어쓰지 않고 그 값으로 새 모델 (2020 확인은 그때)

In [17]:
B4_NOW, LGB_NOW = dict(B4_P), dict(LGB_P)  # 지금 값 (대조군)
yr_all = fp["GYEAR"].to_numpy()
i_tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (yr_all <= fc.VAL_YEAR - 1))
i19 = np.flatnonzero(val)
_B4C = {}

def b4_all(half_life, alpha, hand_k, season_f):  # 2017–2020 초구 전부의 B4 확률 (n, 9), 같은 값은 한 번만
    key = (half_life, alpha, hand_k, season_f)
    if key not in _B4C:
        P = np.full((len(fp), K), np.nan)
        for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
            k = yr_all == yr
            P[k] = b4_online(fp[k], yr, half_life, alpha, hand_k, season_f)
        _B4C[key] = P
    return _B4C[key]

def init_of(P):  # B4 확률 -> LightGBM 출발점 (공마다 평균 0)
    I = np.log(np.clip(P, 1e-6, None))
    return I - I.mean(1, keepdims=True)

def score(b4p, lp):  # 2017–18 학습, 2019 조기 종료 -> (2019 log loss, 트리 수) (model_07 2-3 방식)
    I = init_of(b4_all(**b4p))
    dtr = lgb.Dataset(X_all.iloc[i_tr], y_all[i_tr], init_score=I[i_tr], free_raw_data=False)
    dva = lgb.Dataset(X_all.iloc[i19], y_all[i19], init_score=I[i19], reference=dtr)
    bst = lgb.train({**BASE, **lp}, dtr, 5000, valid_sets=[dva], callbacks=[lgb.early_stopping(100, verbose=False)])
    raw = bst.predict(X_all.iloc[i19], raw_score=True, num_iteration=bst.best_iteration) + I[i19]
    return -log_softmax(raw, axis=1)[np.arange(len(i19)), y_all[i19]].mean(), bst.best_iteration

ll_now, it_now = score(B4_NOW, LGB_NOW)
print(f"지금 값 (대조군): 2019 {ll_now:.5f}, 트리 {it_now} (model_07 2-3 선택값 1.0621·730과 같아야 함)")
assert abs(ll_now - 1.0621) < 1e-4 and it_now == 730

지금 값 (대조군): 2019 1.06210, 트리 730 (model_07 2-3 선택값 1.0621·730과 같아야 함)


### 7-1. B4 (LightGBM은 지금 값)

In [18]:
import time
t0 = time.time()
GRID_B4 = {"half_life": [25.0, 35.4, 50.0], "alpha": [5.0, 7.5, 11.0], "hand_k": [3.5, 5.48, 8.0], "season_f": [0.25]}
cands = [dict(half_life=h, alpha=a, hand_k=k, season_f=s) for h in GRID_B4["half_life"] for a in GRID_B4["alpha"]
         for k in GRID_B4["hand_k"] for s in GRID_B4["season_f"]]
cands += [dict(B4_NOW, season_f=s) for s in (0.15, 0.4)]
RES_B4 = {}
def run_b4(cs):
    for c in cs:
        key = tuple(c.values())
        if key not in RES_B4:
            RES_B4[key] = score(c, LGB_NOW)
run_b4(cands)
for rnd in range(3):  # 1등이 범위 끝이면 그 방향으로 넓힘 (나머지는 1등 값)
    best = dict(zip(B4_NOW, min(RES_B4, key=lambda k: RES_B4[k][0])))
    add = []
    for p in B4_NOW:
        vals = sorted({k[list(B4_NOW).index(p)] for k in RES_B4})
        if best[p] == vals[0]:
            add.append(dict(best, **{p: round(best[p] / 1.5, 3)}))
        elif best[p] == vals[-1]:
            add.append(dict(best, **{p: round(min(best[p] * 1.5, 1.0) if p == "season_f" else best[p] * 1.5, 3)}))
    add = [a for a in add if tuple(a.values()) not in RES_B4]
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    run_b4(add)
tab_b4 = pd.DataFrame([{**dict(zip(B4_NOW, k)), "2019 log loss": v[0], "트리 수": v[1]} for k, v in RES_B4.items()]).sort_values("2019 log loss")
print(f"B4 후보 {len(tab_b4)}개, {time.time() - t0:.0f}초")
display(tab_b4.head(12).round(5))
B4_BEST = dict(zip(B4_NOW, min(RES_B4, key=lambda k: RES_B4[k][0])))
print("B4 1등:", B4_BEST, f"{RES_B4[tuple(B4_BEST.values())][0]:.5f} / 지금 값 {ll_now:.5f} (차이 {RES_B4[tuple(B4_BEST.values())][0] - ll_now:+.5f})")

B4 후보 29개, 203초


,half_life,alpha,hand_k,season_f,2019 log loss,트리 수
13,35.4,7.5,5.48,0.25,1.06210,730
10,35.4,5.0,5.48,0.25,1.06227,730
11,35.4,5.0,8.00,0.25,1.06235,730
16,35.4,11.0,5.48,0.25,1.06242,730
15,35.4,11.0,3.50,0.25,1.06262,730
14,35.4,7.5,8.00,0.25,1.06265,730
23,50.0,7.5,8.00,0.25,1.06266,730
25,50.0,11.0,5.48,0.25,1.06267,730
27,35.4,7.5,5.48,0.15,1.06267,730
12,35.4,7.5,3.50,0.25,1.06276,730


B4 1등: {'half_life': 35.4, 'alpha': 7.5, 'hand_k': 5.48, 'season_f': 0.25} 1.06210 / 지금 값 1.06210 (차이 +0.00000)


### 7-2. LightGBM (7-1 1등 B4 위)

In [19]:
t0 = time.time()
GRID_LGB = {"num_leaves": [7, 15], "min_data_in_leaf": [20, 50, 200], "lambda_l2": [300.0, 1000.0, 3000.0]}
RES_LGB = {}
def run_lgb(cs):
    for c in cs:
        key = (c["num_leaves"], c["min_data_in_leaf"], c["lambda_l2"])
        if key not in RES_LGB:
            RES_LGB[key] = score(B4_BEST, c)
run_lgb([dict(num_leaves=a, min_data_in_leaf=b, lambda_l2=c) for a in GRID_LGB["num_leaves"] for b in GRID_LGB["min_data_in_leaf"] for c in GRID_LGB["lambda_l2"]])
for rnd in range(2):  # 1등이 범위 끝이면 넓힘 (model_07과 같음: lambda_l2 ×3, 잎 ×2)
    b = min(RES_LGB, key=lambda k: RES_LGB[k][0])
    add = []
    if b[2] == max(k[2] for k in RES_LGB):
        add.append(dict(num_leaves=b[0], min_data_in_leaf=b[1], lambda_l2=b[2] * 3))
    if b[0] == max(k[0] for k in RES_LGB):
        add.append(dict(num_leaves=b[0] * 2, min_data_in_leaf=b[1], lambda_l2=b[2]))
    if not add:
        break
    print(f"{rnd + 1}번째 넓히기:", add)
    run_lgb(add)
tab_lgb = pd.DataFrame([{"num_leaves": k[0], "min_data_in_leaf": k[1], "lambda_l2": k[2], "2019 log loss": v[0], "트리 수": v[1]}
                        for k, v in RES_LGB.items()]).sort_values("2019 log loss").reset_index(drop=True)
print(f"LightGBM 후보 {len(tab_lgb)}개, {time.time() - t0:.0f}초")
display(tab_lgb.head(10).round(5))
TOL = 2e-4
ok = tab_lgb[tab_lgb["2019 log loss"] <= tab_lgb["2019 log loss"].iloc[0] + TOL]
pk = ok.sort_values(["num_leaves", "트리 수"]).iloc[0]  # 평평하면 잎 적게 -> 트리 적게 (model_07과 같음)
LGB_PICK = {"num_leaves": int(pk.num_leaves), "min_data_in_leaf": int(pk.min_data_in_leaf), "lambda_l2": float(pk.lambda_l2)}
ROUNDS_PICK = int(pk["트리 수"])
print("고른 LightGBM:", LGB_PICK, f"트리 {ROUNDS_PICK}, 2019 {pk['2019 log loss']:.5f} / 지금 값(B4·LightGBM 모두) {ll_now:.5f} (차이 {pk['2019 log loss'] - ll_now:+.5f})")

1번째 넓히기: [{'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 9000.0}]
LightGBM 후보 19개, 156초


,num_leaves,min_data_in_leaf,lambda_l2,2019 log loss,트리 수
0,7,20,3000.0,1.06195,1794
1,7,50,3000.0,1.06196,1885
2,7,20,9000.0,1.06197,4820
3,15,50,3000.0,1.06201,1460
4,7,200,3000.0,1.06202,1726
5,15,20,3000.0,1.06203,1493
6,15,200,3000.0,1.06207,1549
7,15,200,1000.0,1.06207,574
8,15,50,1000.0,1.06208,573
9,15,20,1000.0,1.06210,572


고른 LightGBM: {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0} 트리 730, 2019 1.06210 / 지금 값(B4·LightGBM 모두) 1.06210 (차이 +0.00000)


### 7-3. 월 단위 재학습으로 비교 (2019)

- 고른 값(B4 1등 + LightGBM 고른 값·트리 수)과 지금 값을 model_09과 같은 월 단위 재학습으로 -> 2019 구종 log loss, 경기 단위 부트스트랩

In [20]:
def monthly_2019(b4p, lp, rounds):  # 월 단위 재학습 2019 공별 log loss
    I = init_of(b4_all(**b4p))
    iy = i19
    out = np.full(len(iy), np.nan)
    for mon in np.unique(MON[iy]):
        sel = MON[iy] == mon
        te = iy[sel]
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < DATE[te].min()))
        bst = lgb.train({**BASE, **lp}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=I[tr]), rounds)
        raw = bst.predict(X_all.iloc[te], raw_score=True) + I[te]
        out[sel] = -log_softmax(raw, axis=1)[np.arange(len(te)), y_all[te]]
    return out

ll_m_now = monthly_2019(B4_NOW, LGB_NOW, LGB_ROUNDS)
assert abs(ll_m_now.mean() - fc.prob_metrics(P_TYPE_ALL[val], y_type[val])["logloss"]) < 1e-6  # = model_09 2019
ll_m_pick = monthly_2019(B4_BEST, LGB_PICK, ROUNDS_PICK)
d = ll_m_now - ll_m_pick  # 양수 = 고른 값이 좋음
s = pd.DataFrame({"g": q_val["gameID"].to_numpy(), "d": d}).groupby("g")["d"].agg(["sum", "count"])
bi = np.random.default_rng(0).integers(0, len(s), (1000, len(s)))
lo, hi = np.percentile(s["sum"].to_numpy()[bi].sum(1) / s["count"].to_numpy()[bi].sum(1), [2.5, 97.5])
print(f"월 단위 2019: 지금 값 {ll_m_now.mean():.5f} / 고른 값 {ll_m_pick.mean():.5f}, 이득 ×1000 {d.mean() * 1000:+.2f} (95% {lo * 1000:+.2f} ~ {hi * 1000:+.2f})")
print("고른 값:", B4_BEST, LGB_PICK, f"트리 {ROUNDS_PICK}")

월 단위 2019: 지금 값 1.06129 / 고른 값 1.06129, 이득 ×1000 +0.00 (95% +0.00 ~ +0.00)
고른 값: {'half_life': 35.4, 'alpha': 7.5, 'hand_k': 5.48, 'season_f': 0.25} {'num_leaves': 7, 'min_data_in_leaf': 20, 'lambda_l2': 1000.0} 트리 730


#### 🔍 7장 결과
#### 2단계 끝 확인: 지금 값 그대로 (B4·LightGBM 모두 1등이거나 평평한 구간 안) -> model_09 그대로
- 대조군 재현: 2019 1.06210, 트리 730 (model_07 2-3과 같음)
- 7-1 B4 (29개): **지금 값(반감 35.4·alpha 7.5·hand_k 5.48·시즌 초 0.25)이 1등** -> 최종 구종 기준으로 골라도 model_06에서 B4 단독으로 고른 값과 같음, 범위 끝 아님
  - 2등 alpha 5 (+0.00017), 반감 50·시즌 초 0.15는 +0.0006 정도 나쁨
- 7-2 LightGBM (19개, λ 9000까지 넓힘): 1등 잎 7·잎당 20·λ 3000 (트리 1,794개) 1.06195 vs 지금 값 1.06210 -> 차이 0.00015로 평평한 구간(0.0002) 안 -> 규칙대로 트리 적은 지금 값 (model_07 2-3과 같은 결론)
- 7-3 월 단위 재학습: 고른 값 = 지금 값이라 차이 0
- 덧: B4 후보 29개 모두 트리 수가 730으로 같음 -> 확인해 보니 조기 종료 곡선에 730 근처 작은 골이 있음 (700구 1.06217 -> 730구 1.06210 -> 760구 1.06218, 다른 B4 값에서도 같은 위치) -> 고정된 bagging 순서(seed 0)가 만드는 골로 보임, 크기 0.00007로 평평한 구간 안이라 결론에 영향 없음
- 위치는 2단계 동안 바뀌지 않아 대상 아님
